<a href="https://colab.research.google.com/github/woooyoung/python_CLI_board_26_08/blob/main/%ED%8C%8C%EC%9D%B4%EC%8D%AC_CLI_board.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datetime import datetime
from dataclasses import dataclass

@dataclass
class Article:
  id: int
  regDate: str
  updateDate: str
  memberId: int
  title: str
  body: str

@dataclass
class Member:
  id: int
  regDate: str
  updateDate: str
  loginId: str
  loginPw: str
  name: str

def now():
    # 현재 시각을 "YYYY-MM-DD HH:MM:SS" 문자열로
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def parse_id(cmd):
    # "article delete 3" -> 3 을 뽑아준다. 잘못된 명령이면 None
    cmdBits = cmd.split(" ")
    if len(cmdBits) < 3:
        print("명령어를 확인해주세요(길이 모자람)")
        return None
    if not cmdBits[2].isdigit():
        print("명령어를 확인해주세요(번호대신 str 입력)")
        return None
    return int(cmdBits[2])

def find_article(articles, article_id):
    # id가 맞는 글을 돌려준다. 없으면 None (found 플래그가 필요 없어짐)
    for article in articles:
        if article.id == article_id:
            return article
    return None

def format_date(regDate):
    # 오늘 쓴 글이면 시:분:초, 이전 글이면 연-월-일 로 보여준다
    today = now().split(" ")[0]        # "2026-09-01"
    datePart = regDate.split(" ")[0]   # 글의 날짜
    timePart = regDate.split(" ")[1]   # 글의 시각
    if datePart == today:
        return timePart
    else:
        return datePart

def make_article_TestData():
  return [
      Article(1, "2025-12-12 12:12:12", "2025-12-12 12:12:12", 1, "제목1", "내용1"),
      Article(2, now(), now(), 1, "제목2", "내용2"),
      Article(3, now(), now(), 2, "제목3", "내용3")
          ]

def make_member_TestData():
  return [
      Member(1, now(), now(), "test1", "test1", "회원1"),
      Member(2, now(), now(), "test2", "test2", "회원2")
          ]

# ===== 메인 =====

print("== CLI 게시판 실행 ==")

last_article_id = 3
last_member_id = 2
articles = make_article_TestData()
members = make_member_TestData()

loginedMember = None

while True:
    cmd = input("명령어 ) ").strip()

    if cmd == "exit":
        break
    elif cmd == "member logout":

      if loginedMember is None:
        print("로그인 하고 이용해")
        continue

      loginedMember = None

      print("로그아웃 성공!")

    elif cmd == "member login":

      if loginedMember is not None:
        print("이미 로그인중인데?")
        continue

      loginId = input("로그인 아이디 : ")
      loginPw = input("비밀번호 : ")

      member = None

      for m in members:
        if m.loginId == loginId:
          member = m
          break

      if member is None:
        print("너 누구임? 일치하는 아이디 없어")
        continue

      if member.loginPw != loginPw:
        print("비번 틀렸어")
        continue

      loginedMember = member

      print("{}님, 로그인 성공!".format(member.name))

    elif cmd == "member join":

      if loginedMember is not None:
        print("이미 로그인중인데?")
        continue

      last_member_id += 1
      # 중복검사 -> 이런애가 나한테 있었나?
      while True:
        loginId = input("로그인 아이디 : ")

        isDup = False

        for member in members:
          if member.loginId == loginId:
            isDup = True
            break

        if isDup:
          print("이미 누가 쓰던데?")
          continue
        break

      # 비밀번호 확인 -> 비밀번호랑 비밀번호 확인이랑 일치하는가?
      while True:
        loginPw = input("비밀번호 : ")
        loginPwConfirm = input("비밀번호 확인: ")
        if loginPw != loginPwConfirm:
          print("비번 불일치")
          continue
        break
      name = input("이름 : ")
      member = Member(last_member_id, now(), now(), loginId, loginPw, name)
      members.append(member)
      print("{}번 회원이 가입되었습니다".format(last_member_id))
      print(members)

    elif cmd == "article write":

      if loginedMember is None:
        print("로그인 하고 이용해")
        continue

      last_article_id += 1
      title = input("제목 : ")
      body = input("내용 : ")
      article = Article(last_article_id, now(), now() ,loginedMember.id, title, body)
      articles.append(article)
      print("{}번 글이 생성되었습니다".format(last_article_id))

    elif cmd == "article list":
      if not articles:
          print("게시글이 없습니다")
      else:
          print("=========================================")
          print("   번호      /     제목       /     내용   /     작성일   ")
          for a in reversed(articles):
              print("   {}      /     {}       /     {}          /     {}   ".format(
                  a.id, a.title, a.body, format_date(a.regDate)))
          print("=========================================")

    elif cmd.startswith("article delete"):

      if loginedMember is None:
        print("로그인 하고 이용해")
        continue

      deletedId = parse_id(cmd)
      if deletedId is None:
          continue

      article = find_article(articles, deletedId)
      if article is None:
          print("{}번 글은 없습니다".format(deletedId))
          continue

      if article.memberId != loginedMember.id:
        print("이거 니꺼 아니야. 권한이 없어")
        continue

      articles.remove(article)
      print("{}번 글이 삭제되었습니다".format(deletedId))

    elif cmd.startswith("article edit"):

      if loginedMember is None:
        print("로그인 하고 이용해")
        continue

      editedId = parse_id(cmd)
      if editedId is None:
          continue

      article = find_article(articles, editedId)
      if article is None:
          print("{}번 글은 없습니다".format(editedId))
          continue

      if article.memberId != loginedMember.id:
        print("이거 니꺼 아니야. 권한이 없어")
        continue

      print("기존 제목 : {}".format(article.title))
      print("기존 내용 : {}".format(article.body))
      article.title = input("새 제목 : ")
      article.body = input("새 내용 : ")
      article.updateDate = now()
      print("{}번 글이 수정되었습니다".format(editedId))

    elif cmd.startswith("article detail"):
      detailId = parse_id(cmd)
      if detailId is None:
          continue

      article = find_article(articles, detailId)
      if article is None:
          print("{}번 글은 없습니다".format(detailId))
          continue

      print("번호 : {}".format(article.id))
      print("작성 날짜 : {}".format(article.regDate))
      print("수정 날짜 : {}".format(article.updateDate))
      print("작성자 번호 : {}".format(article.memberId))
      print("제목 : {}".format(article.title))
      print("내용 : {}".format(article.body))

    else:
      print("지원하지 않는 기능입니다")

print("== CLI 게시판 종료 ==")

== CLI 게시판 실행 ==
명령어 ) article list
   번호      /     제목       /     내용   /     작성일   
   3      /     제목3       /     내용3          /     06:23:55   
   2      /     제목2       /     내용2          /     06:23:55   
   1      /     제목1       /     내용1          /     2025-12-12   
명령어 ) article delete 2
로그인 하고 이용해
명령어 ) article edit 1
로그인 하고 이용해
명령어 ) member login 
로그인 아이디 : test1
비밀번호 : test1
회원1님, 로그인 성공!
명령어 ) article delete 3
이거 니꺼 아니야. 권한이 없어
명령어 ) artciel delete 4
지원하지 않는 기능입니다
명령어 ) article delete 4
4번 글은 없습니다
명령어 ) article delete 2
2번 글이 삭제되었습니다
명령어 ) article list
   번호      /     제목       /     내용   /     작성일   
   3      /     제목3       /     내용3          /     06:23:55   
   1      /     제목1       /     내용1          /     2025-12-12   
명령어 ) member logout
로그아웃 성공!
명령어 ) member login
로그인 아이디 : test2
비밀번호 : test2
회원2님, 로그인 성공!
명령어 ) article delete 1
이거 니꺼 아니야. 권한이 없어
명령어 ) article edit 1
이거 니꺼 아니야. 권한이 없어
명령어 ) article edit 3
기존 제목 : 제목3
기존 내용 : 내용3
새 제목 : xcbv
새 내용 : xcvb
3번 글이 수정